# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [2]:
!gdown https://drive.google.com/uc?id=1i9hkXIusOdNkCTlO0RdL4aN2M7_3FejY

Downloading...
From: https://drive.google.com/uc?id=1i9hkXIusOdNkCTlO0RdL4aN2M7_3FejY
To: /content/co2_emissions.csv
100% 11.9k/11.9k [00:00<00:00, 32.7MB/s]


In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [4]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [5]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
df = df[df['Region'] == 'Asia']

# Let's apply the principles

highlight = 'China'

# Build colour map: highlight in blue, everything else grey
color_map = {c: '#2E75B6' if c == highlight else '#DDDDDD' for c in df['Country'].unique()}

fig = px.line(df, x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''})

fig.update_traces(
    line=dict(width=1.5),   # default: thin for context countries
    showlegend=False      # direct label replaces legend (Gestalt proximity)
)

# Override highlighted country: thicker line
fig.update_traces(
    line=dict(width=3),
    selector=dict(name=highlight)
)

# Direct label at end of highlighted line
last = df.loc[(df['Country'] == highlight) & (df['Year'] == df['Year'].max())]

fig.add_annotation(
    x=last['Year'].values[0], y=last['CO2_Mt'].values[0],
    text=f'<b>{highlight}</b>', showarrow=False,
    xanchor='left', xshift=6,
    font=dict(color='#2E75B6', size=12, family='Arial')
)

fig.update_layout(
    title="China's CO2 emissions tripled 2000–2022 — no other country comes close",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE', title='CO2 Emissions (Mt)'),
    xaxis=dict(showgrid=False, title=''),
    margin=dict(l=60, r=80, t=55, b=40)
)

fig.show()

---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [17]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Reload the original df for Task 2 as it was filtered in Task 1
df = pd.read_csv('co2_emissions.csv')

# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE

sg = df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index().copy()

# Filter sg to only 2000 and 2022 for the slopegraph itself and for calculating color
sg_filtered = sg[sg['Year'].isin([2000, 2022])]

# Create a pivot table to easily compare 2000 and 2022 values for each region
sg_compare = sg_filtered.pivot(index='Region', columns='Year', values='CO2_Mt').reset_index()
sg_compare.columns.name = None # Remove the 'Year' column name from the index
sg_compare = sg_compare.rename(columns={2000: 'CO2_2000', 2022: 'CO2_2022'})

# Now create the color_map based on increase or decrease
color_map = {row['Region']: '#2E75B6' if row['CO2_2000'] < row['CO2_2022'] else '#E07B00' for _, row in sg_compare.iterrows()}

# Sort regions by their 2022 value for better readability
region_order = sg_compare.sort_values('CO2_2022', ascending=False)['Region'].tolist()

fig = px.line(sg_filtered, x='Year', y='CO2_Mt', color='Region',
              color_discrete_map=color_map, markers=True,
              labels={'CO2_Mt': '', 'Year': ''},
              category_orders={'Region': region_order})

for region_name in region_order:
    d = sg_filtered.loc[sg_filtered['Region'] == region_name].sort_values('Year')
    fig.update_traces(
        selector=dict(name=region_name),
        mode='lines+markers+text',
        text=[f'{d["CO2_Mt"].iloc[0]:.0f}', f'{region_name}<br>{d["CO2_Mt"].iloc[1]:.0f}'],
        textposition=['middle left', 'middle right'],
        textfont=dict(size=10, color=color_map[region_name], family='Arial'),
        showlegend=False
    )

fig.update_layout(
    title='Asia shows biggest increase and North America the biggest decrease',
    xaxis=dict(tickvals=[2000, 2022], ticktext=['2000', '2022'],
               showgrid=False, range=[1995, 2028]),
    yaxis=dict(showgrid=False, showticklabels=False, title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=80, r=120, t=55, b=40),
    height=800, width=1000
)

fig.show()
